# Binned-fit farm results

This notebook analyzes the one-row-per-job artifacts produced by `examples/binned-identity/` and `examples/binned-unfolded/`. For every bootstrap row it selects the **lowest finite final $\chi^2$ among the completed repeat checkpoints currently available**. It does not require all 20 repeats, or all 500 rows, to have finished.

A `*_partial.pt` file is restart state rather than a final fit: its recorded best loss need not correspond to its latest parameter vector. Partial checkpoints are counted and reported, but never silently used as physics results. Row 0 is the nominal histogram; rows 1--499 form the bootstrap ensemble.

The workflow is: inventory all requested campaigns, choose one active event-size/detector case, inspect individual rows with the Section 13 diagnostics, then form best-fit bootstrap bands. The final campaign table is also the starting point for later identity-versus-unfolded and smearing comparisons.

In [ ]:
from __future__ import annotations

import gc
import importlib.util
import math
import shutil
import sys
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as py
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from scipy import stats
from tqdm.auto import tqdm

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'examples' / 'binned-identity').is_dir():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / 'src'))

from quantom.binned_results import (
    discover_binned_cases, load_best_for_row, load_fit_checkpoint,
    scan_fit_artifacts, select_best_available,
)

USE_TEX = shutil.which('latex') is not None
mpl.rcParams.update({
    'text.usetex': USE_TEX,
    'text.latex.preamble': r'\usepackage{amsmath}',
    'font.family': 'serif',
    'font.size': 18,
    'axes.labelsize': 30,
    'legend.fontsize': 18,
    'xtick.labelsize': 18,
    'ytick.labelsize': 18,
})
TARGETS = ('p', 'n')
PARAMETER_LABELS = tuple(
    f'{flavor}_{name}'
    for flavor in ('g', 'up', 'dp')
    for name in ('A', 'a', 'b', 'c', 'd')
)
print(f'Repository: {REPO_ROOT}')
print(f'LaTeX rendering: {USE_TEX}')

## 1. Analysis controls

`CASE_IDS_TO_SCAN=None` inventories every case whose artifact directory exists. The first pass opens new or changed `.pt` files; later passes use `analysis/artifact_inventory.csv`. Set it to `(ACTIVE_CASE_ID,)` for a quick single-case pass. The full theory reconstruction is intentionally performed only for the active case.

In [ ]:
ACTIVE_CASE_ID = 'identity-500000'
CASE_IDS_TO_SCAN = None  # None means all cases with an artifact directory
REFRESH_ARTIFACT_INVENTORY = False

DIAGNOSTIC_ROWS = (0,)
Q2_REQUESTS = (1.6384, 10.01)
ENSEMBLE_INTERVAL = (0.05, 0.95)
MAX_ENSEMBLE_ROWS = None  # None uses every selected bootstrap row
RUN_ENSEMBLE_EVOLUTION = True
SAVE_FIGURES = False
FIGURE_FORMATS = ('png', 'pdf')

NOTEBOOK_OUT_DIR = REPO_ROOT / 'notebooks' / 'binned_fit_results'
NOTEBOOK_OUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Discover and inventory campaigns

The table below is derived from the configuration files, not from directory-name guesses. Thus event size, identity versus unfolded data, and unfolded smearing width remain tied to the exact fit configuration.

In [ ]:
case_table = discover_binned_cases(REPO_ROOT)
display(case_table[[
    'case_id', 'event_size', 'detector', 'n_bins',
    'expected_bootstrap_rows', 'expected_repeats', 'artifact_dir_exists',
]])
if ACTIVE_CASE_ID not in set(case_table['case_id']):
    raise KeyError(f'Unknown ACTIVE_CASE_ID={ACTIVE_CASE_ID!r}')

In [ ]:
requested_case_ids = (
    set(case_table.loc[case_table['artifact_dir_exists'], 'case_id'])
    if CASE_IDS_TO_SCAN is None else set(CASE_IDS_TO_SCAN)
)
requested_case_ids.add(ACTIVE_CASE_ID)
campaigns = {}
campaign_rows = []

for case in case_table.itertuples(index=False):
    if case.case_id not in requested_case_ids:
        continue
    print(f'Inventorying {case.case_id}: {case.artifact_dir}', flush=True)
    inventory = scan_fit_artifacts(
        case.artifact_dir, refresh=REFRESH_ARTIFACT_INVENTORY, progress=True
    )
    available = select_best_available(
        inventory,
        expected_bootstrap_rows=case.expected_bootstrap_rows,
        expected_repeats=case.expected_repeats,
    )
    dof = 2 * int(case.n_bins) - 15
    available['dof'] = dof
    available['best_chi2_over_dof'] = available['best_chi2'] / dof
    analysis_dir = Path(case.artifact_dir) / 'analysis'
    analysis_dir.mkdir(parents=True, exist_ok=True)
    available.to_csv(analysis_dir / 'best_repeat_by_row.csv', index=False)
    campaigns[case.case_id] = {
        'case': case, 'inventory': inventory, 'available': available,
        'analysis_dir': analysis_dir, 'dof': dof,
    }
    statuses = available['selection_status'].value_counts()
    selected = available[available['selected']]
    campaign_rows.append({
        'case_id': case.case_id, 'event_size': case.event_size,
        'detector': case.detector, 'dof': dof,
        'complete_repeat_checkpoints': int((inventory['checkpoint_kind'] == 'complete').sum()) if len(inventory) else 0,
        'partial_checkpoints': int((inventory['checkpoint_kind'] == 'partial').sum()) if len(inventory) else 0,
        'selected_rows': len(selected),
        'fully_complete_rows': int(statuses.get('all_repeats_complete', 0)),
        'best_of_available_rows': int(statuses.get('best_of_available', 0)),
        'partial_only_rows': int(statuses.get('partial_only', 0)),
        'missing_rows': int(statuses.get('no_artifact', 0)),
        'median_completed_repeats': float(selected['n_complete_repeats'].median()) if len(selected) else np.nan,
        'median_chi2_over_dof': float(selected['best_chi2_over_dof'].median()) if len(selected) else np.nan,
    })

campaign_summary = pd.DataFrame(campaign_rows).sort_values(
    ['event_size', 'detector'], kind='stable'
).reset_index(drop=True)
campaign_summary.to_csv(NOTEBOOK_OUT_DIR / 'campaign_summary.csv', index=False)
display(campaign_summary)

## 3. Active-case coverage and best-repeat selection

The coverage plot distinguishes optimizer convergence from artifact availability. A selected repeat need not have `success=True`: finite fits stopped by a TNC limit remain available, and the lowest final $\chi^2$ is selected as requested.

In [ ]:
if ACTIVE_CASE_ID not in campaigns:
    raise FileNotFoundError(
        f'No artifact directory was scanned for {ACTIVE_CASE_ID}; '
        'check ACTIVE_CASE_ID and CASE_IDS_TO_SCAN.'
    )
active = campaigns[ACTIVE_CASE_ID]
active_case = active['case']
repeat_inventory = active['inventory']
best_available = active['available']
selected_best = best_available[best_available['selected']].copy()
active_analysis_dir = active['analysis_dir']

print(f'Active case: {ACTIVE_CASE_ID}')
print(f'Artifacts: {active_case.artifact_dir}')
print(f'Selected rows: {len(selected_best)}/{active_case.expected_bootstrap_rows}')
display(best_available['selection_status'].value_counts().rename('rows').to_frame())
display(selected_best.head(15))

In [ ]:
def style_axis(ax, *, xlabel=None, ylabel=None, logx=False, logy=False):
    if xlabel is not None:
        ax.set_xlabel(xlabel, size=30)
    if ylabel is not None:
        ax.set_ylabel(ylabel, size=30)
    if logx:
        ax.set_xscale('log')
    if logy:
        ax.set_yscale('log')
    ax.tick_params(
        direction='in', labelsize=18, which='both', axis='both',
        top=False, right=False,
    )


def save_figure(fig, stem):
    if not SAVE_FIGURES:
        return
    for extension in FIGURE_FORMATS:
        fig.savefig(
            active_analysis_dir / f'{stem}.{extension}',
            dpi=250, bbox_inches='tight',
        )


def plot_campaign_coverage():
    table = best_available
    bootstraps = selected_best[selected_best['bootstrap_row'] > 0]
    nrows, ncols = 2, 2
    fig, axes = py.subplots(
        nrows=nrows, ncols=ncols, figsize=(8 * ncols, 6 * nrows)
    )
    axes = axes.ravel()
    axes[0].plot(table['bootstrap_row'], table['n_complete_repeats'], '.', color='C0', ms=5)
    axes[0].axhline(active_case.expected_repeats, color='black', linestyle=':', linewidth=1.2)
    style_axis(axes[0], xlabel=r'$\mathrm{bootstrap\ row}$', ylabel=r'$N_{\rm completed\ repeats}$')

    if len(bootstraps):
        axes[1].hist(bootstraps['best_chi2_over_dof'], bins=35, density=True, color='C0', alpha=0.65)
        axes[1].axvline(1.0, color='black', linestyle=':', linewidth=1.2)
        axes[2].hist(bootstraps['best_repeat'], bins=np.arange(active_case.expected_repeats + 1) - 0.5, color='C1', alpha=0.75)
    style_axis(axes[1], xlabel=r'$\chi^2_{\rm best}/\nu$', ylabel=r'$\mathrm{density}$')
    style_axis(axes[2], xlabel=r'$\mathrm{selected\ repeat}$', ylabel=r'$\mathrm{rows}$')

    status_counts = table['selection_status'].value_counts()
    axes[3].bar(np.arange(len(status_counts)), status_counts.values, color='C2', alpha=0.75)
    axes[3].set_xticks(np.arange(len(status_counts)), [rf'$\mathrm{{{name.replace("_", r"\_")}}}$' for name in status_counts.index], rotation=25, ha='right')
    style_axis(axes[3], ylabel=r'$\mathrm{rows}$')
    fig.suptitle(
        rf'$\mathrm{{{ACTIVE_CASE_ID.replace("-", r"\! - ")}}},\;\nu={active["dof"]}$', size=24
    )
    fig.subplots_adjust(left=0.09, right=0.99, bottom=0.10, top=0.91, wspace=0.25, hspace=0.32)
    save_figure(fig, 'campaign_coverage')
    py.show()


plot_campaign_coverage()

## 4. Repeat-level optimizer diagnostics

These plots load only the completed repeats for a requested row. Gray curves are other available random initializations; blue marks the repeat selected by final $\chi^2$. The partial repeat, if present, is summarized in the coverage table and left for job resumption.

In [ ]:
def scalar_history(checkpoint, key):
    return pd.DataFrame([
        {name: value for name, value in item.items() if name not in ('parameters', 'gradient')}
        for item in checkpoint.get(key, [])
    ])


def completed_repeats_for_row(bootstrap_row):
    rows = repeat_inventory[
        (repeat_inventory['bootstrap_row'] == bootstrap_row)
        & (repeat_inventory['checkpoint_kind'] == 'complete')
        & repeat_inventory['readable']
        & repeat_inventory['finite_chi2']
    ].sort_values('repeat')
    return rows


def plot_repeat_stability(bootstrap_row):
    rows = completed_repeats_for_row(bootstrap_row)
    if rows.empty:
        print(f'row {bootstrap_row}: no completed finite repeat')
        return
    selected_repeat = int(
        best_available.loc[best_available['bootstrap_row'] == bootstrap_row, 'best_repeat'].iloc[0]
    )
    nrows, ncols = 1, 2
    fig, axes = py.subplots(nrows=nrows, ncols=ncols, figsize=(8 * ncols, 6 * nrows))
    axes[0].plot(rows['repeat'], rows['chi2'], 'o-', color='C0', label=r'$\chi^2_{p+n}$')
    axes[0].axhline(active['dof'], color='black', linestyle=':', label=rf'$\nu={active["dof"]}$')
    chosen = rows[rows['repeat'] == selected_repeat]
    axes[0].plot(chosen['repeat'], chosen['chi2'], '*', color='red', ms=14, label=r'$\mathrm{selected}$')
    axes[0].legend(fontsize=18)
    axes[1].plot(rows['repeat'], rows['chi2_p'], 'o-', color='C1', label=r'$p$')
    axes[1].plot(rows['repeat'], rows['chi2_n'], 's--', color='C2', label=r'$n$')
    axes[1].legend(fontsize=18)
    style_axis(axes[0], xlabel=r'$\mathrm{random\ initialization}$', ylabel=r'$\chi^2$')
    style_axis(axes[1], xlabel=r'$\mathrm{random\ initialization}$', ylabel=r'$\chi^2_{\rm target}$')
    fig.suptitle(rf'$\mathrm{{bootstrap\ row}}={bootstrap_row}$', size=24)
    fig.subplots_adjust(left=0.10, right=0.99, bottom=0.16, top=0.86, wspace=0.22)
    save_figure(fig, f'row_{bootstrap_row:03d}_repeat_stability')
    py.show()


def plot_optimizer_traces(bootstrap_row):
    rows = completed_repeats_for_row(bootstrap_row)
    if rows.empty:
        return
    selected_repeat = int(
        best_available.loc[best_available['bootstrap_row'] == bootstrap_row, 'best_repeat'].iloc[0]
    )
    nrows, ncols = 2, 2
    fig, axes = py.subplots(nrows=nrows, ncols=ncols, figsize=(8 * ncols, 6 * nrows))
    for record in rows.itertuples(index=False):
        checkpoint = load_fit_checkpoint(record.path)
        evaluations = scalar_history(checkpoint, 'evaluation_history')
        iterations = scalar_history(checkpoint, 'iteration_history')
        selected = int(record.repeat) == selected_repeat
        color, alpha, width = ('C0', 1.0, 1.8) if selected else ('0.65', 0.55, 0.9)
        label = rf'$\mathrm{{repeat}}\;{int(record.repeat)}\;\mathrm{{(selected)}}$' if selected else None
        if len(evaluations):
            axes[0, 0].plot(evaluations['evaluation'], evaluations['chi2'], color=color, alpha=alpha, linewidth=width, label=label)
            axes[0, 1].plot(evaluations['evaluation'], evaluations['gradient_inf'], color=color, alpha=alpha, linewidth=width)
        if len(iterations):
            axes[1, 0].plot(iterations['iteration'], iterations['chi2'], color=color, alpha=alpha, linewidth=width)
            axes[1, 1].plot(iterations['iteration'], iterations['gradient_inf'], color=color, alpha=alpha, linewidth=width)
        del checkpoint
    style_axis(axes[0, 0], xlabel=r'$\mathrm{function\ evaluation}$', ylabel=r'$\chi^2$', logy=True)
    style_axis(axes[0, 1], xlabel=r'$\mathrm{function\ evaluation}$', ylabel=r'$|\nabla\chi^2|_\infty$', logy=True)
    style_axis(axes[1, 0], xlabel=r'$\mathrm{accepted\ TNC\ iteration}$', ylabel=r'$\chi^2$', logy=True)
    style_axis(axes[1, 1], xlabel=r'$\mathrm{accepted\ TNC\ iteration}$', ylabel=r'$|\nabla\chi^2|_\infty$', logy=True)
    axes[0, 0].legend(fontsize=16)
    fig.suptitle(rf'$\mathrm{{bootstrap\ row}}={bootstrap_row}:\;\mathrm{{optimizer\ traces}}$', size=24)
    fig.subplots_adjust(left=0.10, right=0.99, bottom=0.09, top=0.92, wspace=0.24, hspace=0.28)
    save_figure(fig, f'row_{bootstrap_row:03d}_optimizer_traces')
    py.show()

## 5. Reconstruct fitted and NNPDF truth models

One JAMX context is built for the active configuration and reused. Each selected checkpoint restores its 15-parameter `BetaPolyPlusPDF`. The truth is the NNPDF member recorded by the configuration, evaluated on the same grid and through the same proton/neutron DIS models.

In [ ]:
FIT_SCRIPT = REPO_ROOT / 'examples' / 'binned-identity' / 'binned_chi2_fit.py'
spec = importlib.util.spec_from_file_location('binned_chi2_fit_analysis', FIT_SCRIPT)
fit_module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = fit_module
spec.loader.exec_module(fit_module)

from jamx import params as aux
from quantom.dis import dis_factory
from quantom.misc.configs import RuntimeConfig
from quantom.pdfs.factory import pdf_factory

fit_conf = RuntimeConfig.from_json(active_case.config_path)
if selected_best.empty:
    raise RuntimeError('The active case has no completed finite repeat checkpoint.')
reference_row = int(selected_best.iloc[0]['bootstrap_row'])
reference_checkpoint = load_best_for_row(best_available, reference_row)
reference_stat_u = reference_checkpoint.get('stat_u')
if reference_stat_u is None:
    reference_stat_u = {
        target: np.sqrt(reference_checkpoint['nominal_counts'][target])
        / reference_checkpoint['n_events'] * reference_checkpoint['norms'][target]
        for target in TARGETS
    }
fit_settings = reference_checkpoint['fit_settings']
fit_problem = fit_module.BinnedChi2Problem(
    fit_conf, reference_checkpoint['bins'],
    reference_checkpoint['nominal_counts'], reference_stat_u,
    reference_checkpoint['norms'], reference_checkpoint['n_events'],
    int(fit_settings['degree']), float(fit_settings['q2_panel_log_span']),
)

if fit_conf.truth is None or fit_conf.truth.kind != 'nnpdf':
    raise ValueError('Section 13 comparison requires an NNPDF truth configuration.')
truth_pdf_config = fit_conf.pdfs.model_copy(update={
    'type': 'NNPDF',
    'kwargs': {
        'nx': fit_conf.grid.ncells[0] + 1,
        'xlims': tuple(float(value) for value in fit_conf.grid.lims[0]),
        'member': int(fit_conf.truth.member),
    },
})
truth_pdfs = pdf_factory(truth_pdf_config, None).to(dtype=torch.float64)
truth_pdf = truth_pdfs[0]
truth_models = {}
for target in TARGETS:
    target_conf = next(item for item in fit_conf.dataset.targets if item.name == target)
    truth_models[target] = dis_factory(
        truth_pdfs, target_conf, fit_problem.context, clip=True
    ).to(dtype=torch.float64).eval()
with torch.no_grad():
    truth_binned = {
        target: fit_problem.bin_cross_sections(truth_models[target]).cpu().numpy()
        for target in TARGETS
    }

MODEL_CACHE = {}
def models_for_row(bootstrap_row):
    if bootstrap_row not in MODEL_CACHE:
        checkpoint = load_best_for_row(best_available, bootstrap_row)
        pdf, models = fit_problem.build_models(int(checkpoint['seed']))
        pdf.load_state_dict(checkpoint['model_state_dict'])
        pdf.eval()
        for model in models.values():
            model.eval()
        MODEL_CACHE[bootstrap_row] = (checkpoint, pdf, models)
    return MODEL_CACHE[bootstrap_row]


def evolved_pdf_values(pdf, model, requested_q2):
    x_grid = model._gridx
    q2_grid = model.ctx.grid.axes[1]
    q2_index = int(torch.argmin(torch.abs(q2_grid - requested_q2)))
    with torch.no_grad():
        values = model.evo(pdf(x_grid))
    return (
        x_grid.detach().cpu().numpy(), float(q2_grid[q2_index]),
        {
            'g': values.g[:, q2_index].detach().cpu().numpy(),
            'up': values.qp.u[:, q2_index].detach().cpu().numpy(),
            'dp': values.qp.d[:, q2_index].detach().cpu().numpy(),
        },
    )


def fixed_q2_x_grid(q2, n_points=250):
    target_conf = fit_conf.dataset.targets[0]
    rs2 = float(target_conf.rs) ** 2
    xmin = q2 / (float(target_conf.ycut) * (rs2 - float(aux.M2)))
    xmax = q2 / (q2 + float(target_conf.W2cut) - float(aux.M2))
    xmin = max(xmin, float(fit_problem.grid.bounds[0, 0]))
    xmax = min(xmax, float(fit_problem.grid.bounds[0, 1]))
    return np.geomspace(xmin * (1.0 + 1e-10), xmax * (1.0 - 1e-10), n_points)

print(f'Built active fit context and NNPDF member {fit_conf.truth.member} truth.')

## 6. Section 13 diagnostics for selected rows

For every row in `DIAGNOSTIC_ROWS`, this reproduces the useful Section 13 views: repeat stability and optimizer histories; PDFs at the initial and approximately $10\,\mathrm{GeV}^2$ scales; proton and neutron differential cross sections at those scales; and binned data, fit, truth, fit/data, and pulls.

In [ ]:
def plot_pdf_fit(bootstrap_row, requested_q2):
    _, fitted_pdf, fitted_models = models_for_row(bootstrap_row)
    x_fit, q2_actual, fitted = evolved_pdf_values(fitted_pdf, fitted_models['p'], requested_q2)
    x_truth, q2_truth, truth = evolved_pdf_values(truth_pdf, truth_models['p'], requested_q2)
    if not np.isclose(q2_actual, q2_truth):
        raise RuntimeError('fit and truth selected different Q2 grid points')
    names = (('up', r'$xu^+$'), ('dp', r'$xd^+$'), ('g', r'$xg$'))
    nrows, ncols = 1, 3
    fig, axes = py.subplots(nrows=nrows, ncols=ncols, figsize=(8 * ncols, 6 * nrows))
    for ax, (name, ylabel) in zip(axes, names, strict=True):
        ax.plot(x_truth, x_truth * truth[name], color='black', linewidth=1.5, label=rf'$\mathrm{{NNPDF\ member\ {fit_conf.truth.member}}}$')
        ax.plot(x_fit, x_fit * fitted[name], color='C0', linewidth=1.5, label=r'$\mathrm{BetaPoly\ fit}$')
        style_axis(ax, xlabel=r'$x$', ylabel=ylabel, logx=True)
    axes[0].legend(fontsize=18)
    fig.suptitle(rf'$\mathrm{{row}}={bootstrap_row},\;Q^2={q2_actual:.4g}\;\mathrm{{GeV}}^2$', size=24)
    fig.subplots_adjust(left=0.07, right=0.99, bottom=0.15, top=0.86, wspace=0.24)
    save_figure(fig, f'row_{bootstrap_row:03d}_pdf_Q2_{q2_actual:g}')
    py.show()


def plot_differential_xsec_fit(bootstrap_row, target, requested_q2):
    _, _, fitted_models = models_for_row(bootstrap_row)
    q2_grid = fitted_models[target].ctx.grid.axes[1]
    q2_index = int(torch.argmin(torch.abs(q2_grid - requested_q2)))
    q2_actual = float(q2_grid[q2_index])
    x = fixed_q2_x_grid(q2_actual)
    points = torch.tensor(np.column_stack((x, np.full_like(x, q2_actual))), dtype=torch.float64)
    with torch.no_grad():
        fitted = fitted_models[target](points).detach().cpu().numpy()
        truth = truth_models[target](points).detach().cpu().numpy()
    nrows, ncols = 1, 2
    fig, axes = py.subplots(nrows=nrows, ncols=ncols, figsize=(8 * ncols, 6 * nrows))
    axes[0].plot(x, truth, color='black', linewidth=1.5, label=r'$\mathrm{NNPDF\ truth}$')
    axes[0].plot(x, fitted, color='C0', linewidth=1.5, label=r'$\mathrm{BetaPoly\ fit}$')
    axes[0].legend(fontsize=18)
    axes[1].plot(x, fitted / truth, color='C0', linewidth=1.5)
    axes[1].axhline(1.0, color='black', linestyle=':', linewidth=1.0)
    style_axis(axes[0], xlabel=r'$x$', ylabel=r'$d^2\sigma/(dx\,dQ^2)$', logx=True, logy=True)
    style_axis(axes[1], xlabel=r'$x$', ylabel=r'$\mathrm{fit}/\mathrm{truth}$', logx=True)
    fig.suptitle(rf'$\mathrm{{row}}={bootstrap_row},\;{target},\;Q^2={q2_actual:.4g}\;\mathrm{{GeV}}^2$', size=24)
    fig.subplots_adjust(left=0.10, right=0.99, bottom=0.15, top=0.86, wspace=0.22)
    save_figure(fig, f'row_{bootstrap_row:03d}_{target}_differential_Q2_{q2_actual:g}')
    py.show()


def plot_binned_fit(bootstrap_row, target):
    checkpoint, _, _ = models_for_row(bootstrap_row)
    counts = np.asarray(checkpoint['counts'][target], dtype=float)
    data = counts / checkpoint['n_events'] * checkpoint['norms'][target]
    stat_u = np.asarray(checkpoint.get('stat_u', reference_stat_u)[target], dtype=float)
    fitted = np.asarray(checkpoint['predictions'][target], dtype=float)
    truth = truth_binned[target]
    indices = np.arange(len(truth))
    fit_over_data = np.divide(fitted, data, out=np.full_like(fitted, np.nan), where=data > 0)
    relative_data_u = np.divide(stat_u, data, out=np.full_like(stat_u, np.nan), where=data > 0)
    pulls = (data - fitted) / stat_u
    nrows, ncols = 2, 2
    fig, axes = py.subplots(nrows=nrows, ncols=ncols, figsize=(8 * ncols, 6 * nrows))
    axes = axes.ravel()
    axes[0].errorbar(indices, data, yerr=stat_u, fmt='o', color='black', ms=4, capsize=2, label=r'$\sigma_i^{\rm data}$')
    axes[0].plot(indices, truth, 's-', color='C1', linewidth=1.2, label=r'$T_i^{\rm NNPDF}$')
    axes[0].plot(indices, fitted, '^-', color='C0', linewidth=1.2, label=r'$T_i^{\rm fit}$')
    axes[0].legend(fontsize=16)
    axes[1].errorbar(indices, data / truth, yerr=stat_u / truth, fmt='o', color='black', ms=4, capsize=2, label=r'$\mathrm{data}/\mathrm{truth}$')
    axes[1].plot(indices, fitted / truth, '^-', color='C0', linewidth=1.2, label=r'$\mathrm{fit}/\mathrm{truth}$')
    axes[1].axhline(1.0, color='black', linestyle=':', linewidth=1.0)
    axes[1].legend(fontsize=16)
    axes[2].fill_between(indices, 1.0 - relative_data_u, 1.0 + relative_data_u, color='0.75', alpha=0.5, step='mid', label=r'$\mathrm{data\ stat.\ uncertainty}$')
    axes[2].plot(indices, fit_over_data, '^-', color='C0', linewidth=1.2, label=r'$T_i^{\rm fit}/\sigma_i^{\rm data}$')
    axes[2].axhline(1.0, color='black', linestyle=':', linewidth=1.0)
    axes[2].legend(fontsize=16)
    axes[3].plot(indices, pulls, 'o', color='C0')
    axes[3].axhline(0.0, color='black', linestyle=':', linewidth=1.0)
    axes[3].axhline(1.0, color='gray', linestyle='--', linewidth=1.0)
    axes[3].axhline(-1.0, color='gray', linestyle='--', linewidth=1.0)
    style_axis(axes[0], xlabel=r'$\mathrm{bin\ index}$', ylabel=r'$\sigma_i$', logy=True)
    style_axis(axes[1], xlabel=r'$\mathrm{bin\ index}$', ylabel=r'$\mathrm{ratio\ to\ NNPDF\ truth}$')
    style_axis(axes[2], xlabel=r'$\mathrm{bin\ index}$', ylabel=r'$\mathrm{fit}/\mathrm{data}$')
    style_axis(axes[3], xlabel=r'$\mathrm{bin\ index}$', ylabel=r'$(\sigma_i^{\rm data}-T_i^{\rm fit})/\mathrm{stat}_{u,i}$')
    chi2_target = float(checkpoint['chi2_by_target'][target])
    fig.suptitle(rf'$\mathrm{{row}}={bootstrap_row},\;{target},\;\chi^2_{{{target}}}={chi2_target:.2f}$', size=24)
    fig.subplots_adjust(left=0.09, right=0.99, bottom=0.10, top=0.91, wspace=0.24, hspace=0.28)
    save_figure(fig, f'row_{bootstrap_row:03d}_{target}_binned_fit')
    py.show()

In [ ]:
available_diagnostic_rows = [
    row for row in DIAGNOSTIC_ROWS
    if bool(best_available.loc[best_available['bootstrap_row'] == row, 'selected'].iloc[0])
]
for bootstrap_row in available_diagnostic_rows:
    plot_repeat_stability(bootstrap_row)
    plot_optimizer_traces(bootstrap_row)
    for requested_q2 in Q2_REQUESTS:
        plot_pdf_fit(bootstrap_row, requested_q2)
        for target in TARGETS:
            plot_differential_xsec_fit(bootstrap_row, target, requested_q2)
    for target in TARGETS:
        plot_binned_fit(bootstrap_row, target)

## 7. Best-fit bootstrap ensemble

The ensemble uses rows 1 onward; row 0 is retained as the nominal reference. First we collect stored bin predictions and fitted parameters, which requires no new theory integration. The 5th--95th percentile bands are empirical across the selected bootstrap fits.

In [ ]:
ensemble_selection = selected_best[selected_best['bootstrap_row'] > 0].sort_values('bootstrap_row')
if MAX_ENSEMBLE_ROWS is not None:
    ensemble_selection = ensemble_selection.iloc[:MAX_ENSEMBLE_ROWS]
ensemble_rows = ensemble_selection['bootstrap_row'].to_numpy(dtype=int)
ensemble_parameters = []
ensemble_predictions = {target: [] for target in TARGETS}
for record in tqdm(ensemble_selection.itertuples(index=False), total=len(ensemble_selection), desc='Loading selected fits'):
    checkpoint = load_fit_checkpoint(record.best_checkpoint)
    ensemble_parameters.append(np.asarray(checkpoint['final_parameters'], dtype=float))
    for target in TARGETS:
        ensemble_predictions[target].append(np.asarray(checkpoint['predictions'][target], dtype=float))
    del checkpoint
ensemble_parameters = np.asarray(ensemble_parameters)
ensemble_predictions = {target: np.asarray(values) for target, values in ensemble_predictions.items()}
ensemble_summary = ensemble_selection[[
    'bootstrap_row', 'best_repeat', 'best_seed', 'best_success',
    'best_chi2', 'best_chi2_p', 'best_chi2_n', 'best_chi2_over_dof',
    'n_complete_repeats', 'n_partial_checkpoints', 'selection_status',
]].copy()
if len(ensemble_parameters):
    for index, label in enumerate(PARAMETER_LABELS):
        ensemble_summary[f'parameter_{label}'] = ensemble_parameters[:, index]
    ensemble_summary.to_csv(active_analysis_dir / 'best_fit_ensemble_summary.csv', index=False)
    np.savez_compressed(
        active_analysis_dir / 'best_fit_binned_ensemble.npz',
        bootstrap_rows=ensemble_rows, parameters=ensemble_parameters,
        predictions_p=ensemble_predictions['p'], predictions_n=ensemble_predictions['n'],
        truth_p=truth_binned['p'], truth_n=truth_binned['n'],
        bins=np.asarray(reference_checkpoint['bins']),
    )
print(f'Bootstrap fits in ensemble: {len(ensemble_rows)}')
display(ensemble_summary.head())

In [ ]:
def ensemble_quantiles(values):
    low, high = ENSEMBLE_INTERVAL
    return np.quantile(values, (low, 0.5, high), axis=0)


def plot_binned_prediction_ensemble(target):
    values = ensemble_predictions[target]
    if not len(values):
        return
    low, median, high = ensemble_quantiles(values)
    truth = truth_binned[target]
    indices = np.arange(len(truth))
    nominal = load_best_for_row(best_available, 0) if bool(best_available.loc[0, 'selected']) else reference_checkpoint
    nominal_data = np.asarray(nominal['counts'][target]) / nominal['n_events'] * nominal['norms'][target]
    nominal_u = np.asarray(nominal.get('stat_u', reference_stat_u)[target])
    nrows, ncols = 1, 1
    fig = py.figure(figsize=(8 * ncols, 6 * nrows))
    grid = fig.add_gridspec(2, 1, height_ratios=(3, 1), hspace=0.05)
    ax = fig.add_subplot(grid[0])
    ratio_ax = fig.add_subplot(grid[1], sharex=ax)
    ax.fill_between(indices, low, high, color='C0', alpha=0.35, label=rf'${int(100*(ENSEMBLE_INTERVAL[1]-ENSEMBLE_INTERVAL[0]))}\%\;\mathrm{{bootstrap\ range}}$')
    ax.plot(indices, median, color='C0', linewidth=1.8, label=r'$\mathrm{bootstrap\ median}$')
    ax.plot(indices, truth, color='black', linewidth=1.5, label=r'$\mathrm{NNPDF\ truth}$')
    ax.errorbar(indices, nominal_data, yerr=nominal_u, fmt='o', color='C3', ms=3.5, elinewidth=1.0, capsize=0, label=r'$\mathrm{nominal\ data}$')
    ratios = values / truth[None, :]
    ratio_low, ratio_median, ratio_high = ensemble_quantiles(ratios)
    ratio_ax.fill_between(indices, ratio_low, ratio_high, color='C0', alpha=0.35)
    ratio_ax.plot(indices, ratio_median, color='C0', linewidth=1.8)
    ratio_ax.errorbar(indices, nominal_data / truth, yerr=nominal_u / truth, fmt='o', color='C3', ms=3.5, elinewidth=1.0, capsize=0)
    ratio_ax.axhline(1.0, color='black', linestyle=':', linewidth=1.0)
    ax.legend(fontsize=15, ncols=2)
    style_axis(ax, ylabel=r'$\sigma_{\rm bin}$', logy=True)
    style_axis(ratio_ax, xlabel=r'$\mathrm{bin\ index}$', ylabel=r'$\mathrm{fit}/\mathrm{truth}$')
    ax.tick_params(labelbottom=False)
    fig.suptitle(rf'$\mathrm{{{ACTIVE_CASE_ID}}},\;{target}$', size=24)
    fig.subplots_adjust(left=0.13, right=0.99, bottom=0.13, top=0.88)
    save_figure(fig, f'ensemble_binned_{target}')
    py.show()


def plot_parameter_ensemble():
    if not len(ensemble_parameters):
        return
    nrows, ncols = 3, 5
    fig, axes = py.subplots(nrows=nrows, ncols=ncols, figsize=(8 * ncols, 6 * nrows))
    for index, (ax, label) in enumerate(zip(axes.flat, PARAMETER_LABELS, strict=True)):
        ax.hist(ensemble_parameters[:, index], bins=35, color='C0', alpha=0.72)
        ax.axvline(np.median(ensemble_parameters[:, index]), color='black', linestyle='--', linewidth=1.2)
        style_axis(ax, xlabel=rf'${label.replace("_", r"_{") + "}"}$', ylabel=r'$\mathrm{replicas}$')
    fig.suptitle(r'$\mathrm{BetaPoly\ parameters\ across\ selected\ bootstrap\ fits}$', size=25)
    fig.subplots_adjust(left=0.055, right=0.995, bottom=0.07, top=0.94, hspace=0.34, wspace=0.28)
    save_figure(fig, 'ensemble_parameter_histograms')
    py.show()


for target in TARGETS:
    plot_binned_prediction_ensemble(target)
plot_parameter_ensemble()

## 8. Evolved PDF and fixed-$Q^2$ cross-section bands

This is the compute-heavy post-processing cell. It reconstructs one fitted model per selected bootstrap row, evolves it once on the full grid, and evaluates proton and neutron cross sections at the requested $Q^2$ values. Set `MAX_ENSEMBLE_ROWS` for a pilot or `RUN_ENSEMBLE_EVOLUTION=False` when only the stored binned predictions are needed.

In [ ]:
pdf_ensemble = {}
differential_ensemble = {}
q2_actual_by_request = {}

if RUN_ENSEMBLE_EVOLUTION and len(ensemble_rows):
    q2_grid = fit_problem.context.grid.axes[1]
    for requested_q2 in Q2_REQUESTS:
        q2_index = int(torch.argmin(torch.abs(q2_grid - requested_q2)))
        q2_actual = float(q2_grid[q2_index])
        q2_actual_by_request[requested_q2] = q2_actual
        pdf_ensemble[requested_q2] = {name: [] for name in ('g', 'up', 'dp')}
        for target in TARGETS:
            differential_ensemble[requested_q2, target] = []

    for index, (record, parameters) in enumerate(
        tqdm(zip(ensemble_selection.itertuples(index=False), ensemble_parameters), total=len(ensemble_rows), desc='Evolving selected fits')
    ):
        pdf, models = fit_problem.build_models(int(record.best_seed))
        with torch.no_grad():
            tuple(pdf.parameters())[0].copy_(torch.as_tensor(parameters.reshape(3, 5), dtype=torch.float64))
        pdf.eval()
        for model in models.values():
            model.eval()
        x_pdf = models['p']._gridx
        with torch.no_grad():
            evolved = models['p'].evo(pdf(x_pdf))
            for requested_q2 in Q2_REQUESTS:
                q2_actual = q2_actual_by_request[requested_q2]
                q2_index = int(torch.argmin(torch.abs(q2_grid - q2_actual)))
                pdf_ensemble[requested_q2]['g'].append(evolved.g[:, q2_index].cpu().numpy())
                pdf_ensemble[requested_q2]['up'].append(evolved.qp.u[:, q2_index].cpu().numpy())
                pdf_ensemble[requested_q2]['dp'].append(evolved.qp.d[:, q2_index].cpu().numpy())
                x_diff = fixed_q2_x_grid(q2_actual)
                points = torch.tensor(np.column_stack((x_diff, np.full_like(x_diff, q2_actual))), dtype=torch.float64)
                for target in TARGETS:
                    differential_ensemble[requested_q2, target].append(models[target](points).cpu().numpy())
        del evolved, models, pdf
        if (index + 1) % 25 == 0:
            gc.collect()

    x_pdf_ensemble = x_pdf.detach().cpu().numpy()
    pdf_ensemble = {request: {name: np.asarray(values) for name, values in flavors.items()} for request, flavors in pdf_ensemble.items()}
    differential_ensemble = {key: np.asarray(values) for key, values in differential_ensemble.items()}
    evolved_export = {
        'bootstrap_rows': ensemble_rows, 'x_pdf': x_pdf_ensemble,
    }
    for requested_q2 in Q2_REQUESTS:
        q2_tag = f'{q2_actual_by_request[requested_q2]:.8g}'.replace('.', 'p')
        evolved_export[f'Q2_{q2_tag}'] = np.asarray(q2_actual_by_request[requested_q2])
        for flavor in ('g', 'up', 'dp'):
            evolved_export[f'pdf_{flavor}_Q2_{q2_tag}'] = pdf_ensemble[requested_q2][flavor]
        for target in TARGETS:
            evolved_export[f'differential_{target}_Q2_{q2_tag}'] = differential_ensemble[requested_q2, target]
    np.savez_compressed(active_analysis_dir / 'best_fit_evolved_ensemble.npz', **evolved_export)
else:
    print('Evolved ensemble was not requested.')

In [ ]:
def plot_pdf_ensemble(requested_q2):
    if requested_q2 not in pdf_ensemble:
        return
    q2_actual = q2_actual_by_request[requested_q2]
    _, _, truth = evolved_pdf_values(truth_pdf, truth_models['p'], q2_actual)
    names = (('up', r'$xu^+$'), ('dp', r'$xd^+$'), ('g', r'$xg$'))
    nrows, ncols = 1, 3
    fig = py.figure(figsize=(8 * ncols, 6 * nrows))
    grid = fig.add_gridspec(2, ncols, height_ratios=(3, 1), hspace=0.05, wspace=0.25)
    main_axes = [fig.add_subplot(grid[0, index]) for index in range(ncols)]
    ratio_axes = [fig.add_subplot(grid[1, index], sharex=main_axes[index]) for index in range(ncols)]
    for index, (name, ylabel) in enumerate(names):
        values = x_pdf_ensemble[None, :] * pdf_ensemble[requested_q2][name]
        truth_values = x_pdf_ensemble * truth[name]
        low, median, high = ensemble_quantiles(values)
        ratios = values / truth_values[None, :]
        ratio_low, ratio_median, ratio_high = ensemble_quantiles(ratios)
        ax, ratio_ax = main_axes[index], ratio_axes[index]
        ax.fill_between(x_pdf_ensemble, low, high, color='C0', alpha=0.35, label=r'$\mathrm{bootstrap\ range}$')
        ax.plot(x_pdf_ensemble, median, color='C0', linewidth=1.8, label=r'$\mathrm{bootstrap\ median}$')
        ax.plot(x_pdf_ensemble, truth_values, color='black', linewidth=1.5, label=r'$\mathrm{NNPDF\ truth}$')
        ratio_ax.fill_between(x_pdf_ensemble, ratio_low, ratio_high, color='C0', alpha=0.35)
        ratio_ax.plot(x_pdf_ensemble, ratio_median, color='C0', linewidth=1.8)
        ratio_ax.axhline(1.0, color='black', linestyle=':', linewidth=1.0)
        style_axis(ax, ylabel=ylabel, logx=True)
        style_axis(ratio_ax, xlabel=r'$x$', ylabel=r'$\mathrm{fit}/\mathrm{truth}$', logx=True)
        ax.tick_params(labelbottom=False)
    main_axes[0].legend(fontsize=15)
    fig.suptitle(rf'$Q^2={q2_actual:.4g}\;\mathrm{{GeV}}^2$', size=24)
    fig.subplots_adjust(left=0.07, right=0.99, bottom=0.12, top=0.88)
    save_figure(fig, f'ensemble_pdfs_Q2_{q2_actual:g}')
    py.show()


def plot_differential_ensemble(requested_q2, target):
    key = (requested_q2, target)
    if key not in differential_ensemble:
        return
    q2_actual = q2_actual_by_request[requested_q2]
    x = fixed_q2_x_grid(q2_actual)
    points = torch.tensor(np.column_stack((x, np.full_like(x, q2_actual))), dtype=torch.float64)
    with torch.no_grad():
        truth = truth_models[target](points).cpu().numpy()
    values = differential_ensemble[key]
    low, median, high = ensemble_quantiles(values)
    ratio_low, ratio_median, ratio_high = ensemble_quantiles(values / truth[None, :])
    nrows, ncols = 1, 2
    fig, axes = py.subplots(nrows=nrows, ncols=ncols, figsize=(8 * ncols, 6 * nrows))
    axes[0].fill_between(x, low, high, color='C0', alpha=0.35, label=r'$\mathrm{bootstrap\ range}$')
    axes[0].plot(x, median, color='C0', linewidth=1.8, label=r'$\mathrm{bootstrap\ median}$')
    axes[0].plot(x, truth, color='black', linewidth=1.5, label=r'$\mathrm{NNPDF\ truth}$')
    axes[0].legend(fontsize=15)
    axes[1].fill_between(x, ratio_low, ratio_high, color='C0', alpha=0.35)
    axes[1].plot(x, ratio_median, color='C0', linewidth=1.8)
    axes[1].axhline(1.0, color='black', linestyle=':', linewidth=1.0)
    style_axis(axes[0], xlabel=r'$x$', ylabel=r'$d^2\sigma/(dx\,dQ^2)$', logx=True, logy=True)
    style_axis(axes[1], xlabel=r'$x$', ylabel=r'$\mathrm{fit}/\mathrm{truth}$', logx=True)
    fig.suptitle(rf'${target},\;Q^2={q2_actual:.4g}\;\mathrm{{GeV}}^2$', size=24)
    fig.subplots_adjust(left=0.10, right=0.99, bottom=0.15, top=0.86, wspace=0.22)
    save_figure(fig, f'ensemble_{target}_differential_Q2_{q2_actual:g}')
    py.show()


for requested_q2 in Q2_REQUESTS:
    plot_pdf_ensemble(requested_q2)
    for target in TARGETS:
        plot_differential_ensemble(requested_q2, target)

## 9. Cross-campaign comparison starting point

This deliberately stays high-level until the scientific comparison choices are fixed. It shows coverage and median best-fit quality versus event count for every scanned case; `campaign_summary.csv` and each case's `analysis/best_repeat_by_row.csv` provide the common tables for the next comparison plots.

In [ ]:
if len(campaign_summary):
    nrows, ncols = 1, 2
    fig, axes = py.subplots(nrows=nrows, ncols=ncols, figsize=(8 * ncols, 6 * nrows))
    for detector, rows in campaign_summary.groupby('detector', sort=True):
        rows = rows.sort_values('event_size')
        label = rf'$\mathrm{{{detector.replace(" ", r"\;").replace("=", r"=")}}}$'
        axes[0].plot(rows['event_size'], rows['selected_rows'], 'o-', linewidth=1.4, label=label)
        axes[1].plot(rows['event_size'], rows['median_chi2_over_dof'], 'o-', linewidth=1.4, label=label)
    axes[0].axhline(500, color='black', linestyle=':', linewidth=1.0)
    axes[1].axhline(1.0, color='black', linestyle=':', linewidth=1.0)
    style_axis(axes[0], xlabel=r'$N_{\rm evt}$', ylabel=r'$N_{\rm selected\ rows}$', logx=True)
    style_axis(axes[1], xlabel=r'$N_{\rm evt}$', ylabel=r'$\mathrm{median}(\chi^2_{\rm best}/\nu)$', logx=True)
    axes[0].legend(fontsize=15)
    fig.subplots_adjust(left=0.10, right=0.99, bottom=0.16, top=0.98, wspace=0.23)
    if SAVE_FIGURES:
        for extension in FIGURE_FORMATS:
            fig.savefig(NOTEBOOK_OUT_DIR / f'campaign_comparison.{extension}', dpi=250, bbox_inches='tight')
    py.show()

## Outputs and interpretation

For every scanned case, `artifacts/binned_chi2/analysis/best_repeat_by_row.csv` records availability and the selected checkpoint path. `artifact_inventory.csv` is the incremental scan cache. For the active case, `best_fit_ensemble_summary.csv`, `best_fit_binned_ensemble.npz`, and (when requested) `best_fit_evolved_ensemble.npz` are compact inputs for later comparisons. The notebook-level `notebooks/binned_fit_results/campaign_summary.csv` compares campaigns. Figures are written beside the active case's tables only when `SAVE_FIGURES=True`.

A row labeled `best_of_available` is scientifically usable for inspection but still has fewer than the configured repeats. A `partial_only` row has restart state but no completed fit and is excluded from every ensemble band. This distinction lets analysis proceed now without disguising incomplete farm coverage.